In [1]:
! pip install -U bitsandbytes -q

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from peft import PeftModel, PeftConfig

# Load the rest port of IDF file.
file_path = "v2_nextpart.idf"
output_path = "v2_final.idf"

# Load the EPlus-LLMv2 config.
peft_model_id = "EPlus-LLM/EPlus-LLMv2"
config = PeftConfig.from_pretrained(peft_model_id)

# Load the base LLM, flan-t5-xxl, and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xxl", load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xxl")

# Load the Lora model
model = PeftModel.from_pretrained(model, peft_model_id)

# Generation config
generation_config = model.generation_config
generation_config.max_new_tokens = 5000
generation_config.temperature = 0.1
generation_config.top_p = 0.1
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

# Please provide your input here — a description of the desired building
# For more details, please refer to the paper: https://doi.org/10.1016/j.autcon.2025.106223
input=f"Simulate a U-shaped building that is 99.73 meters high, with a gable roof. The horizontal segment is 732.31 meters long and 17.54 meters wide. The left vertical segment is 256.31 meters long and 206.96 meters wide. The right vertical segment is 431.54 meters long and 62 meters wide. The roof ridge is 8.77 meters to the length side of the horizontal segment, and 128.16 meters, 215.77 meters to the width side of the vertical segments, respectively. The attic height is 139.71 meters. The building orientation is 62 degrees to the north. The building has 3 thermal zones with each segment as one thermal zone. The window-to-wall ratio is 0.32. The window sill height is 33.91 meters, the window height is 65.82 meters, and the window jamb width is 0.01 meters. The window U-factor is 6.36 W/m2K and the SHGC is 0.89. The wall is made of wood, with a thickness of 0.48 meters and the wall insulation is RSI 1.6 m2K/W, U-factor 0.63 W/m2K. The roof is made of metal, with a thickness of 0.09 meters and the roof insulation is RSI 5.4 m2K/W, U-factor 0.19 W/m2K. The floor is made of concrete, covered with carpet. The ventilation rate is 2.32 ach. The infiltration rate is 0.55 ach. The people density is 16.61 m2/person, the light density is 4.48 W/m2, and the electric equipment density is 22.63 W/m2. Occupancy starts at 7:00 and ends at 18:00. The occupancy rate is 1. The unoccupancy rate is 0.3. The heating setpoint is 21.54 Celsius in occupancy period and 15.86 Celsius in unoccupancy period. The cooling setpoint is 22.6 Celsius in occupancy period and 26.72 Celsius in unoccupancy period."
input_ids = tokenizer(input, return_tensors="pt", truncation=False)
generated_ids = model.generate(input_ids = input_ids.input_ids,
                           attention_mask = input_ids.attention_mask,
                           generation_config = generation_config)
generated_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# print(generated_output)

# Output the building energy model in IDF file
with open(file_path, 'r', encoding='utf-8') as file:
    nextpart = file.read()
final_text = nextpart + "\n\n" + generated_output
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(final_text)
print(f"Building Energy Model Auto-Generated: {output_path}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 160.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 98.12 MiB is free. Process 64046 has 14.64 GiB memory in use. Of the allocated memory 14.01 GiB is allocated by PyTorch, and 528.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)